# Activity 11 — Natural Language Processing: Sentiment Analysis
## AI-Powered Customer Review Analytics with Hugging Face Transformers

**Course:** Introduction to Artificial Intelligence
**Session:** 33 — Natural Language Processing
**Topic:** Tokenization · Rule-Based NLP · Transformer Models · Sentiment Classification

---

## Business Scenario — ReviewIQ / TechNest

You are a junior data scientist at **ReviewIQ**, an AI analytics company whose client **TechNest** — an online electronics retailer — receives thousands of product reviews every day. Manually reading each review to determine whether it is positive or negative costs the support team hours of work. Your job is to build an automated **sentiment analysis pipeline** that:

1. **Tags reviews as Positive or Negative** so the marketing team can surface testimonials.
2. **Flags negative reviews** for priority customer support escalation.
3. **Auto-routes high-confidence predictions** and sends low-confidence reviews to human review.

You will approach this in two stages:

| Task | Approach | Focus |
|------|----------|-------|
| **Task 1** | Rule-based (VADER) vs. Transformer (DistilBERT) | Tokenization, pipeline, model comparison |
| **Task 2** | Full evaluation + business deployment | Confusion matrix, error analysis, threshold tuning |

> **Dataset note:** We use the **IMDB Movie Reviews** benchmark dataset — the standard evaluation corpus for sentiment analysis — as a proxy for product review text. The language patterns (expressing opinions, using hedging or superlatives) are the same whether the subject is a film or a gadget. In a real deployment, ReviewIQ would fine-tune the model on TechNest-specific reviews.


## Setup — Libraries and Configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Hugging Face
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer

# Rule-based NLP
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Evaluation
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display

# Reproducibility
SEED = 11
rng = np.random.default_rng(SEED)

print("All libraries loaded successfully.")
print(f"Random seed: {SEED}")


---
# Task 1 — From Rules to Transformers: Building a Sentiment Classifier

**Goal:** Understand how NLP has evolved from rule-based systems (VADER) to transformer-based models (DistilBERT), and measure the accuracy gap on a real movie-review dataset.

**Pipeline overview:**
1. Load the IMDB dataset and explore review text
2. Understand how BERT tokenizes text
3. Run VADER (rule-based) as a baseline
4. Run DistilBERT (transformer) on the same reviews
5. Compare accuracy and analyze where each approach succeeds or fails


### Step 1.1 — Load the IMDB Dataset

We load the **IMDB Movie Reviews** dataset directly from the Hugging Face Hub using the `datasets` library. The first run downloads ~80 MB and caches it locally; subsequent runs load from cache in seconds.

The dataset contains 50,000 reviews split evenly between positive and negative sentiment. We sample **400 reviews** from the test split — 200 for Task 1 experiments and 200 for the Task 2 full evaluation — using a fixed random seed so every student works with identical data.


In [ ]:
print("Loading IMDB dataset from Hugging Face Hub...")
print("(First run: downloads ~80 MB — takes 20–40 seconds)")
print("(Subsequent runs: loads from local cache — near-instant)\n")

dataset = load_dataset("imdb")

# Sample 400 reviews from the test split with fixed seed
n_total = 400
idx = rng.choice(len(dataset['test']), size=n_total, replace=False)
reviews = [dataset['test'][int(i)]['text'] for i in idx]
labels  = [dataset['test'][int(i)]['label'] for i in idx]  # 0=Negative, 1=Positive

# Helper: truncate long reviews for display (model handles truncation internally)
def truncate(text, n=300):
    return text[:n] + "..." if len(text) > n else text

df = pd.DataFrame({
    'review'      : reviews,
    'label'       : labels,
    'sentiment'   : ['Positive' if l == 1 else 'Negative' for l in labels],
    'review_short': [truncate(r) for r in reviews],
    'word_count'  : [len(r.split()) for r in reviews],
})

print(f"Dataset loaded: {len(df)} reviews")
print(f"  Positive : {(df.label==1).sum()} ({(df.label==1).mean()*100:.1f}%)")
print(f"  Negative : {(df.label==0).sum()} ({(df.label==0).mean()*100:.1f}%)")
print(f"\nReview length statistics:")
print(f"  Mean  : {df.word_count.mean():.0f} words")
print(f"  Median: {df.word_count.median():.0f} words")
print(f"  Max   : {df.word_count.max()} words")

print(f"\nSample reviews:")
for i in range(3):
    row = df.iloc[i]
    print(f"\n  [{row.sentiment.upper()}] {row.review_short[:200]}...")


### Step 1.2 — Tokenization: How BERT Reads Text

Before any transformer model can process text, it must convert words into **tokens** — integer IDs that map to entries in the model's vocabulary. DistilBERT uses **WordPiece tokenization**, which handles unknown words by splitting them into recognizable subword units.

Key special tokens added by BERT-family models:
- `[CLS]` — "Classification" token prepended to every input; its final hidden state is used for sequence-level tasks like sentiment classification.
- `[SEP]` — "Separator" token appended at the end of every sequence.
- `##` prefix — indicates a subword continuation (e.g., "playing" → ["play", "##ing"]).

**Maximum length:** DistilBERT accepts at most **512 tokens**. Reviews longer than this are automatically truncated.


In [ ]:
print("Loading DistilBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
print(f"Vocabulary size : {tokenizer.vocab_size:,} tokens")
print(f"Model max length: {tokenizer.model_max_length} tokens\n")

# ── Example 1: standard sentence ──────────────────────────────────────────
sample = "This product is absolutely brilliant — I could not be happier!"
tokens = tokenizer.tokenize(sample)
encoding = tokenizer(sample, return_tensors='pt')
full_tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])

print("=" * 60)
print("Example 1: Standard sentence")
print("=" * 60)
print(f"Original : {sample}")
print(f"Lowercase: {sample.lower()}")
print(f"\nWordPiece tokens ({len(tokens)}):")
print(f"  {tokens}")
print(f"\nWith special tokens (what the model sees):")
print(f"  {full_tokens}")

# ── Example 2: WordPiece splitting on unusual words ───────────────────────
print("\n" + "=" * 60)
print("Example 2: WordPiece subword splitting")
print("=" * 60)
test_words = ["unimaginably", "electroencephalography", "disappointing", "NLP"]
for word in test_words:
    wt = tokenizer.tokenize(word)
    print(f"  '{word}' → {wt}")

# ── Example 3: Long review truncation ─────────────────────────────────────
print("\n" + "=" * 60)
print("Example 3: Long review truncation")
print("=" * 60)
long_review = df['review'].iloc[df['word_count'].idxmax()]
enc_long = tokenizer(long_review, truncation=True, max_length=512, return_tensors='pt')
print(f"Longest review in sample : {df['word_count'].max()} words")
print(f"After BERT tokenization  : {enc_long['input_ids'].shape[1]} tokens (truncated to 512)")
print(f"\n  → The first 512 tokens are used; content beyond that is discarded.")
print(f"  → This affects ~{(df['word_count'] > 350).sum()} of our {len(df)} sampled reviews.")

# ── Visualization: token length distribution ───────────────────────────────
token_counts = []
for review in df['review'].tolist()[:50]:   # sample 50 for speed
    enc = tokenizer(review, truncation=False, return_tensors='pt')
    token_counts.append(enc['input_ids'].shape[1])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(token_counts, bins=20, color='#5B9BD5', edgecolor='white')
ax.axvline(512, color='#FF4444', linestyle='--', linewidth=2, label='Max length (512)')
ax.set_xlabel("Token count per review")
ax.set_ylabel("Number of reviews")
ax.set_title("Token Length Distribution — 50 Sample IMDB Reviews", fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_token_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nToken count stats (50 reviews):")
print(f"  Mean: {np.mean(token_counts):.0f}  Median: {np.median(token_counts):.0f}  Max: {max(token_counts)}")


### Step 1.3 — Rule-Based Baseline: VADER Sentiment Analyzer

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based lexicon model designed specifically for social media text. It requires no training and runs in milliseconds.

VADER outputs four scores for each text:
- `pos`, `neg`, `neu` — proportion of text that is positive, negative, or neutral (sum to 1.0)
- `compound` — normalized weighted composite score in [−1.0, +1.0]

**Decision rule:** `compound ≥ 0.05` → Positive, otherwise → Negative.

VADER represents the **rule-based / lexicon era** of NLP — still widely used for quick prototyping because it requires no compute and no labeled data.


In [ ]:
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# Work with the first 50 reviews (Task 1 subset)
df_task1 = df.iloc[:50].copy().reset_index(drop=True)
N1 = len(df_task1)

print(f"Running VADER on {N1} reviews...")
vader_rows = []
for review in df_task1['review']:
    scores = sia.polarity_scores(review)
    vader_rows.append({
        'compound': scores['compound'],
        'pos'     : scores['pos'],
        'neg'     : scores['neg'],
        'neu'     : scores['neu'],
        'pred'    : 1 if scores['compound'] >= 0.05 else 0,
    })
vader_df = pd.DataFrame(vader_rows)
df_task1['vader_pred']     = vader_df['pred'].values
df_task1['vader_compound'] = vader_df['compound'].values
df_task1['vader_pos']      = vader_df['pos'].values
df_task1['vader_neg']      = vader_df['neg'].values

vader_acc = (df_task1['vader_pred'] == df_task1['label']).mean()
print(f"VADER Accuracy: {vader_acc*100:.1f}%  (majority baseline: 50.0%)")

# ── Sample predictions ──────────────────────────────────────────────────────
print(f"\n{'Review (first 50 chars)':<52} {'True':>5} {'VADER':>6} {'Compound':>10} {'':>2}")
print("-" * 78)
for _, row in df_task1.iloc[:8].iterrows():
    true_s = "POS" if row['label'] == 1 else "NEG"
    pred_s = "POS" if row['vader_pred'] == 1 else "NEG"
    flag   = "✓" if row['label'] == row['vader_pred'] else "✗"
    txt    = row['review_short'][:48]
    print(f"{txt:<52} {true_s:>5} {pred_s:>6} {row['vader_compound']:>10.3f}  {flag}")

# ── Cases where VADER fails ─────────────────────────────────────────────────
failures = df_task1[df_task1['vader_pred'] != df_task1['label']]
print(f"\nVADER failures: {len(failures)}/{N1} reviews")
print(f"\nTop 3 VADER failures (sorted by |compound| — most confident wrong):")
failures_sorted = failures.reindex(failures['vader_compound'].abs().sort_values(ascending=False).index).head(3)
for i, (_, row) in enumerate(failures_sorted.iterrows(), 1):
    true_s = "Positive" if row['label'] == 1 else "Negative"
    pred_s = "Positive" if row['vader_pred'] == 1 else "Negative"
    print(f"\n  [{i}] True: {true_s} | VADER: {pred_s} | Compound: {row['vader_compound']:.3f}")
    print(f"       {row['review_short'][:180]}...")


### Step 1.4 — Transformer-Based Sentiment: DistilBERT Pipeline

**DistilBERT** (Distilled BERT) is a smaller, faster version of Google's BERT model, retaining 97% of BERT's accuracy at 40% fewer parameters and 60% faster inference. The model we load — `distilbert-base-uncased-finetuned-sst-2-english` — has been **fine-tuned on the Stanford Sentiment Treebank (SST-2)**, so it can predict positive/negative sentiment out of the box with no additional training.

Hugging Face's `pipeline()` abstraction handles the full inference stack:
1. Tokenization
2. Forward pass through the transformer
3. Softmax over the output logits → confidence score
4. Label mapping (POSITIVE / NEGATIVE)

> **First run:** The model (~268 MB) is downloaded automatically and cached locally. **No Hugging Face account or API key is required.** Subsequent runs load from cache.


In [ ]:
print("Loading DistilBERT sentiment pipeline...")
print("First run downloads ~268 MB and may take 30–90 seconds.")
print("Subsequent runs load from local cache (~3 seconds).\n")

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,       # -1 = CPU  (set to 0 for GPU if available)
    truncation=True,
    max_length=512,
)
print("Model ready.\n")

# Run inference on the same 50 reviews used for VADER
print(f"Running inference on {N1} reviews (CPU — expect ~30 seconds)...")
db_results = sentiment_pipe(df_task1['review'].tolist(), batch_size=8)

df_task1['db_label'] = [r['label'] for r in db_results]
df_task1['db_score'] = [r['score'] for r in db_results]
df_task1['db_pred']  = [1 if r['label'] == 'POSITIVE' else 0 for r in db_results]

db_acc = (df_task1['db_pred'] == df_task1['label']).mean()
print(f"\nDistilBERT Accuracy: {db_acc*100:.1f}%  (VADER: {vader_acc*100:.1f}%)")

# ── Sample predictions ──────────────────────────────────────────────────────
print(f"\n{'Review (first 50 chars)':<52} {'True':>5} {'Pred':>6} {'Confidence':>12} {'':>2}")
print("-" * 80)
for _, row in df_task1.iloc[:8].iterrows():
    true_s = "POS" if row['label'] == 1 else "NEG"
    pred_s = "POS" if row['db_pred'] == 1 else "NEG"
    flag   = "✓" if row['label'] == row['db_pred'] else "✗"
    txt    = row['review_short'][:48]
    print(f"{txt:<52} {true_s:>5} {pred_s:>6} {row['db_score']:>12.1%}  {flag}")


### Step 1.5 — VADER vs. DistilBERT: Side-by-Side Comparison

Now we compare the two approaches on the same 50 reviews using two visualizations:
- **Accuracy bar chart** — overall accuracy vs. the 50% majority baseline
- **Agreement breakdown** — where models agree/disagree, and which model is right when they differ


In [ ]:
# ── Agreement categories ────────────────────────────────────────────────────
both_correct   = ((df_task1['vader_pred'] == df_task1['label']) &
                  (df_task1['db_pred']    == df_task1['label'])).sum()
both_wrong     = ((df_task1['vader_pred'] != df_task1['label']) &
                  (df_task1['db_pred']    != df_task1['label'])).sum()
only_vader_ok  = ((df_task1['vader_pred'] == df_task1['label']) &
                  (df_task1['db_pred']    != df_task1['label'])).sum()
only_db_ok     = ((df_task1['vader_pred'] != df_task1['label']) &
                  (df_task1['db_pred']    == df_task1['label'])).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("VADER vs. DistilBERT — Sentiment Comparison (50 IMDB Reviews)",
             fontsize=13, fontweight='bold')

# Panel A: Accuracy bars
ax = axes[0]
models = ['VADER\n(Rule-Based)', 'DistilBERT\n(Transformer)']
accs   = [vader_acc * 100, db_acc * 100]
colors = ['#5B9BD5', '#ED7D31']
bars   = ax.bar(models, accs, color=colors, edgecolor='white', width=0.45)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 1.5,
            f"{acc:.1f}%", ha='center', fontsize=13, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', alpha=0.6, label='Majority baseline (50%)')
ax.set_ylim(0, 115)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title("Overall Accuracy", fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel B: Agreement breakdown
ax2 = axes[1]
cats   = ['Both\nCorrect', 'Both\nWrong', 'VADER only\nCorrect', 'DistilBERT only\nCorrect']
counts = [both_correct, both_wrong, only_vader_ok, only_db_ok]
cols   = ['#70AD47', '#FF4444', '#5B9BD5', '#ED7D31']
b2     = ax2.bar(cats, counts, color=cols, edgecolor='white')
for bar, cnt in zip(b2, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, cnt + 0.3,
             str(cnt), ha='center', fontsize=13, fontweight='bold')
ax2.set_ylabel("Number of Reviews", fontsize=11)
ax2.set_title("Prediction Agreement Breakdown", fontsize=11)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Agreement summary ({N1} reviews):")
print(f"  Both correct:             {both_correct:3d} ({both_correct/N1*100:.1f}%)")
print(f"  Both wrong:               {both_wrong:3d} ({both_wrong/N1*100:.1f}%)")
print(f"  Only VADER correct:       {only_vader_ok:3d} ({only_vader_ok/N1*100:.1f}%)")
print(f"  Only DistilBERT correct:  {only_db_ok:3d} ({only_db_ok/N1*100:.1f}%)")

# ── Disagreement examples ────────────────────────────────────────────────────
disagree = df_task1[df_task1['vader_pred'] != df_task1['db_pred']].head(3)
if len(disagree) > 0:
    print(f"\nExamples where models DISAGREE:")
    for _, row in disagree.iterrows():
        true_s  = "Positive" if row['label'] == 1 else "Negative"
        vader_s = "Positive" if row['vader_pred'] == 1 else "Negative"
        db_s    = row['db_label'].capitalize()
        print(f"\n  True: {true_s} | VADER: {vader_s} | DistilBERT: {db_s} ({row['db_score']:.1%})")
        print(f"  {row['review_short'][:160]}...")


### Step 1.6 — Confidence Score Distribution

Unlike VADER's compound score, DistilBERT outputs a **confidence score** (0.5–1.0) for its predicted label — the probability assigned to the winning class after softmax.

A well-calibrated model should be **more accurate on high-confidence predictions** than on low-confidence ones. This cell visualizes whether that holds true.


In [ ]:
correct_mask     = df_task1['db_pred'] == df_task1['label']
correct_scores   = df_task1.loc[ correct_mask, 'db_score'].values
incorrect_scores = df_task1.loc[~correct_mask, 'db_score'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("DistilBERT Confidence Score Distribution (50 Reviews)",
             fontsize=13, fontweight='bold')

# Panel A: histogram correct vs incorrect
ax1 = axes[0]
bins = np.linspace(0.5, 1.0, 21)
ax1.hist(correct_scores,   bins=bins, alpha=0.75, color='#70AD47',
         label=f'Correct ({len(correct_scores)})', edgecolor='white')
ax1.hist(incorrect_scores, bins=bins, alpha=0.75, color='#FF4444',
         label=f'Incorrect ({len(incorrect_scores)})', edgecolor='white')
ax1.axvline(0.9, color='black', linestyle='--', alpha=0.6, label='0.9 threshold')
ax1.set_xlabel("Confidence Score", fontsize=11)
ax1.set_ylabel("Count", fontsize=11)
ax1.set_title("Correct vs. Incorrect Predictions", fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Panel B: pie of confidence buckets
ax2 = axes[1]
edges  = [0.50, 0.70, 0.90, 1.01]
labels_b = ['Low (0.50–0.70)', 'Medium (0.70–0.90)', 'High (0.90–1.00)']
colors_b = ['#FFC000', '#5B9BD5', '#70AD47']
counts_b = [((df_task1['db_score'] >= lo) & (df_task1['db_score'] < hi)).sum()
            for lo, hi in zip(edges[:-1], edges[1:])]
ax2.pie(counts_b, labels=labels_b, colors=colors_b, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 10})
ax2.set_title("Reviews by Confidence Bucket", fontsize=11)

plt.tight_layout()
plt.savefig('fig_confidence_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confidence statistics:")
print(f"  Correct predictions   — mean confidence: {correct_scores.mean():.3f}")
print(f"  Incorrect predictions — mean confidence: {incorrect_scores.mean():.3f}")
print(f"  Reviews with conf > 0.90: {(df_task1['db_score'] > 0.90).sum()}/{N1}")
print(f"  Reviews with conf > 0.95: {(df_task1['db_score'] > 0.95).sum()}/{N1}")


### Step 1.7 — Interactive Sentiment Analyzer

Use the widget below to test both VADER and DistilBERT on **any text you write**. Try these challenging cases:

- **Sarcasm:** *"Oh great, another product that breaks in a week. Just what I needed."*
- **Mixed sentiment:** *"The camera is phenomenal but the battery life is a disaster."*
- **Negation:** *"This is not bad at all — I was pleasantly surprised."*
- **Formal praise:** *"The engineering is impeccably refined and the build quality is exceptional."*

**Reflection Question Q1** — answer in the Markdown cell below after experimenting.


In [ ]:
def run_live_sentiment(text):
    if not text.strip():
        print("Please enter some text.")
        return

    db_result  = sentiment_pipe([text])[0]
    vader_sc   = sia.polarity_scores(text)
    vader_pred = "POSITIVE" if vader_sc['compound'] >= 0.05 else "NEGATIVE"

    db_color    = '#2e7d32' if db_result['label'] == 'POSITIVE' else '#c62828'
    vader_color = '#2e7d32' if vader_pred == 'POSITIVE' else '#c62828'

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    fig.suptitle(f'"{text[:90]}{"..." if len(text) > 90 else ""}"',
                 fontsize=10, style='italic', y=1.03)

    # DistilBERT panel
    ax1 = axes[0]
    ax1.barh(['Confidence'], [db_result['score']], color=db_color, height=0.35)
    ax1.set_xlim(0, 1)
    ax1.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel("Score")
    ax1.set_title(f"DistilBERT: {db_result['label']} ({db_result['score']:.1%})",
                  fontsize=12, fontweight='bold', color=db_color)
    ax1.grid(axis='x', alpha=0.3)

    # VADER panel
    ax2 = axes[1]
    vcats  = ['Positive', 'Neutral', 'Negative']
    vvals  = [vader_sc['pos'], vader_sc['neu'], vader_sc['neg']]
    vcols  = ['#70AD47', '#5B9BD5', '#FF4444']
    ax2.barh(vcats, vvals, color=vcols, height=0.35, edgecolor='white')
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Score")
    ax2.set_title(f"VADER: {vader_pred}  (compound = {vader_sc['compound']:.3f})",
                  fontsize=12, fontweight='bold', color=vader_color)
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

text_input = widgets.Textarea(
    value='This movie was absolutely fantastic — one of the best I have ever seen!',
    layout=widgets.Layout(width='68%', height='68px'),
)
run_btn = widgets.Button(
    description='Analyze', button_style='primary',
    layout=widgets.Layout(width='120px'),
)
out_widget = widgets.Output()

def on_click(b):
    out_widget.clear_output(wait=True)
    with out_widget:
        run_live_sentiment(text_input.value)

run_btn.on_click(on_click)

print("Step 1.7 — Interactive Sentiment Analyzer")
print("Type any text below and click Analyze.\n")
display(widgets.VBox([
    widgets.Label("Enter text:"),
    text_input,
    run_btn,
    out_widget,
]))


### Reflection Question Q1 — Answer Here

**Q1.** After using the interactive widget in Step 1.7, test the following sentence with both models: *"Oh sure, because waiting 3 weeks for a broken product is everyone's idea of a great shopping experience."*

- What does VADER predict, and what is its compound score?
- What does DistilBERT predict, and what is its confidence?
- Which model is correct for this sentence?
- Why does VADER struggle with this type of language? Use the concept of a **lexicon** in your explanation.

*Answer: …*


---
# Task 2 — Evaluation, Error Analysis & Business Deployment

**Goal:** Rigorously evaluate DistilBERT on 200 held-out IMDB reviews, understand where and why it fails, and design a threshold-based **auto-tagging system** that ReviewIQ can deploy for TechNest.

**Pipeline:**
1. Run inference on 200 fresh reviews
2. Compute confusion matrix and classification report
3. Analyze the most confident wrong predictions
4. Measure accuracy vs. confidence (calibration)
5. Simulate the ReviewIQ business deployment with a tunable threshold


### Step 2.1 — Full Evaluation: 200 Reviews

We use reviews 200–399 from our 400-review sample — a fresh set not touched in Task 1 — to get an unbiased estimate of model accuracy.

> **Runtime:** ~1–2 minutes on CPU with `batch_size=16`.


In [ ]:
df_task2 = df.iloc[200:].copy().reset_index(drop=True)
N2 = len(df_task2)

print(f"Task 2 evaluation set: {N2} reviews")
print(f"  Positive: {(df_task2.label==1).sum()} ({(df_task2.label==1).mean()*100:.1f}%)")
print(f"  Negative: {(df_task2.label==0).sum()} ({(df_task2.label==0).mean()*100:.1f}%)")
print(f"\nRunning DistilBERT inference on {N2} reviews...")
print("(CPU — expect ~60–120 seconds)\n")

results_200 = sentiment_pipe(df_task2['review'].tolist(), batch_size=16)

df_task2['db_label'] = [r['label'] for r in results_200]
df_task2['db_score'] = [r['score'] for r in results_200]
df_task2['db_pred']  = [1 if r['label'] == 'POSITIVE' else 0 for r in results_200]

accuracy = (df_task2['db_pred'] == df_task2['label']).mean()
majority_baseline = max((df_task2.label==1).mean(), (df_task2.label==0).mean())

print(f"DistilBERT Accuracy : {accuracy*100:.1f}%")
print(f"Majority Baseline   : {majority_baseline*100:.1f}%  (always predict majority class)")
print(f"Improvement over baseline: +{(accuracy - majority_baseline)*100:.1f} pp")


### Step 2.2 — Confusion Matrix and Classification Report

The **confusion matrix** breaks down predictions into four categories:
- **TP** (True Positive) — correctly predicted Positive
- **TN** (True Negative) — correctly predicted Negative
- **FP** (False Positive) — predicted Positive, actually Negative
- **FN** (False Negative) — predicted Negative, actually Positive

From these, we compute **Precision** (how often a positive prediction is right), **Recall** (how often real positives are caught), and **F1-Score** (their harmonic mean).


In [ ]:
y_true = df_task2['label'].values
y_pred = df_task2['db_pred'].values

cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred, target_names=['Negative', 'Positive'])
cr_dict = classification_report(y_true, y_pred, target_names=['Negative', 'Positive'],
                                output_dict=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(f"DistilBERT Evaluation — {N2} IMDB Reviews",
             fontsize=13, fontweight='bold')

# Confusion matrix
ax1 = axes[0]
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Negative', 'Positive'])
disp.plot(ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title("Confusion Matrix", fontsize=11)

TN, FP, FN, TP = cm.ravel()
ax1.set_xlabel(f"Predicted label\n\nTN={TN}  FP={FP}  FN={FN}  TP={TP}", fontsize=10)

# Precision / Recall / F1 grouped bars
ax2 = axes[1]
metrics = ['Precision', 'Recall', 'F1-Score']
neg_vals = [cr_dict['Negative']['precision'],
            cr_dict['Negative']['recall'],
            cr_dict['Negative']['f1-score']]
pos_vals = [cr_dict['Positive']['precision'],
            cr_dict['Positive']['recall'],
            cr_dict['Positive']['f1-score']]
x = np.arange(len(metrics))
w = 0.35
b1 = ax2.bar(x - w/2, neg_vals, w, label='Negative', color='#FF6B6B', edgecolor='white')
b2 = ax2.bar(x + w/2, pos_vals, w, label='Positive',  color='#70AD47', edgecolor='white')
for bars, vals in [(b1, neg_vals), (b2, pos_vals)]:
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 0.015,
                 f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(metrics)
ax2.set_ylim(0, 1.18)
ax2.set_ylabel("Score"); ax2.set_title("Precision / Recall / F1 by Class", fontsize=11)
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Classification Report:")
print(cr)
print(f"Confusion matrix values:")
print(f"  TN={TN}  FP={FP}")
print(f"  FN={FN}  TP={TP}")


### Step 2.3 — Error Analysis: What Did the Model Get Wrong?

Accuracy alone does not explain *why* a model fails. Error analysis — inspecting the actual misclassified examples — is how practitioners improve models in practice.

We sort errors by **confidence** (highest first) to find the cases where the model was most certain but most wrong. These "confident failures" often reveal systematic weaknesses: sarcasm, domain mismatch, complex negation, or ambiguous language.


In [ ]:
errors = df_task2[df_task2['db_pred'] != df_task2['label']].copy()
errors_sorted = errors.sort_values('db_score', ascending=False).reset_index(drop=True)

fp_mask = (df_task2['db_pred'] == 1) & (df_task2['label'] == 0)
fn_mask = (df_task2['db_pred'] == 0) & (df_task2['label'] == 1)

print(f"Total errors: {len(errors)}/{N2} ({len(errors)/N2*100:.1f}%)")
print(f"  False Positives (predicted POS, actually NEG): {fp_mask.sum()}")
print(f"  False Negatives (predicted NEG, actually POS): {fn_mask.sum()}")
print(f"\n{'─'*80}")
print("Top 5 most confident WRONG predictions:")
print(f"{'─'*80}")

for i, row in errors_sorted.head(5).iterrows():
    true_s = "Positive" if row['label'] == 1 else "Negative"
    pred_s = row['db_label'].capitalize()
    print(f"\n#{i+1}  True: {true_s}  |  Predicted: {pred_s}  |  Confidence: {row['db_score']:.1%}")
    print(f"    {row['review_short'][:220]}...")

# ── Error types chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
categories = ['True Negatives\n(correct)', 'True Positives\n(correct)',
               'False Positives\n(wrong)', 'False Negatives\n(wrong)']
TN_v, FP_v, FN_v, TP_v = cm.ravel()
counts_pie = [TN_v, TP_v, FP_v, FN_v]
colors_pie = ['#5B9BD5', '#70AD47', '#FF6B6B', '#FFC000']
bars = ax.bar(categories, counts_pie, color=colors_pie, edgecolor='white')
for bar, cnt in zip(bars, counts_pie):
    ax.text(bar.get_x() + bar.get_width()/2, cnt + 0.5,
            str(cnt), ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel("Count")
ax.set_title(f"Prediction Breakdown — {N2} Reviews", fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


### Step 2.4 — Calibration: Accuracy vs. Confidence

A well-calibrated model should have **higher accuracy in the high-confidence bucket** than in the low-confidence bucket. This analysis also informs threshold selection: if the model is consistently wrong in the 0.50–0.70 range, those predictions should be routed to human review rather than auto-tagged.


In [ ]:
bucket_edges  = [0.50, 0.60, 0.70, 0.80, 0.90, 1.01]
bucket_labels = ['0.50–0.60', '0.60–0.70', '0.70–0.80', '0.80–0.90', '0.90–1.00']
bucket_acc, bucket_cnt = [], []

for lo, hi in zip(bucket_edges[:-1], bucket_edges[1:]):
    mask    = (df_task2['db_score'] >= lo) & (df_task2['db_score'] < hi)
    subset  = df_task2[mask]
    acc     = (subset['db_pred'] == subset['label']).mean() if len(subset) > 0 else 0.0
    bucket_acc.append(acc * 100)
    bucket_cnt.append(len(subset))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Calibration Analysis — Accuracy vs. Confidence",
             fontsize=13, fontweight='bold')

x = np.arange(len(bucket_labels))

ax1 = axes[0]
bars1 = ax1.bar(x, bucket_acc, color='#5B9BD5', edgecolor='white')
ax1.set_xticks(x); ax1.set_xticklabels(bucket_labels, rotation=25, ha='right')
ax1.set_ylim(0, 115); ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy by Confidence Bucket")
ax1.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)
for bar, acc, cnt in zip(bars1, bucket_acc, bucket_cnt):
    ax1.text(bar.get_x() + bar.get_width()/2, acc + 1.5,
             f"{acc:.0f}%\n(n={cnt})", ha='center', va='bottom', fontsize=9)

ax2 = axes[1]
bars2 = ax2.bar(x, bucket_cnt, color='#ED7D31', edgecolor='white')
ax2.set_xticks(x); ax2.set_xticklabels(bucket_labels, rotation=25, ha='right')
ax2.set_ylabel("Number of Reviews")
ax2.set_title("Review Count by Confidence Bucket")
ax2.grid(axis='y', alpha=0.3)
for bar, cnt in zip(bars2, bucket_cnt):
    ax2.text(bar.get_x() + bar.get_width()/2, cnt + 0.4,
             str(cnt), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"{'Bucket':<14} {'Count':>7} {'Accuracy':>10} {'Cumulative Coverage':>22}")
print("-" * 57)
cumulative = 0
for lbl, cnt, acc in zip(bucket_labels, bucket_cnt, bucket_acc):
    cumulative += cnt
    print(f"{lbl:<14} {cnt:>7} {acc:>9.1f}% {cumulative/N2*100:>21.1f}%")


### Step 2.5 — Business Deployment: ReviewIQ Auto-Tagging Threshold

ReviewIQ's pipeline works as follows:
- If `confidence ≥ threshold` → **auto-tag** (no human needed)
- If `confidence < threshold` → **route to human review**

Raising the threshold:
- **Increases accuracy** of auto-tagged reviews (only the most certain predictions are auto-tagged)
- **Decreases coverage** (more reviews require expensive human review)

We plot this trade-off curve and identify two operating points:
- **Default (0.50):** maximum coverage — all reviews auto-tagged
- **High-precision (0.90):** high accuracy — only confident predictions are auto-tagged


In [ ]:
thresholds = np.arange(0.50, 1.00, 0.005)
auto_pct, auto_acc_list = [], []

for t in thresholds:
    auto_mask   = df_task2['db_score'] >= t
    auto_subset = df_task2[auto_mask]
    if len(auto_subset) == 0:
        auto_pct.append(0)
        auto_acc_list.append(100.0)
    else:
        acc = (auto_subset['db_pred'] == auto_subset['label']).mean()
        auto_pct.append(len(auto_subset) / N2 * 100)
        auto_acc_list.append(acc * 100)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

l1, = ax1.plot(thresholds, auto_pct,       color='#5B9BD5', lw=2.5, label='% Auto-tagged')
l2, = ax2.plot(thresholds, auto_acc_list,  color='#70AD47', lw=2.5, label='Auto-tag Accuracy')

ax1.axvline(0.50, color='gray',    ls=':',  alpha=0.8)
ax1.axvline(0.90, color='#FF4444', ls='--', alpha=0.8)

ax1.set_xlabel("Confidence Threshold", fontsize=11)
ax1.set_ylabel("% Reviews Auto-tagged",  color='#5B9BD5', fontsize=11)
ax2.set_ylabel("Auto-tag Accuracy (%)",  color='#70AD47', fontsize=11)
ax1.set_ylim(0, 110); ax2.set_ylim(40, 110)

legend_handles = [
    l1, l2,
    mpatches.Patch(color='gray',    label='Default threshold (0.50)'),
    mpatches.Patch(color='#FF4444', label='High-precision threshold (0.90)'),
]
ax1.legend(handles=legend_handles, loc='center right', fontsize=9)
plt.title("ReviewIQ Auto-Tagging Threshold Analysis\nTechNest Customer Review Pipeline",
          fontsize=12, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

print("Operating point comparison:\n")
for t_name, t_val in [('Default  (0.50)', 0.50), ('High-precision (0.90)', 0.90)]:
    m      = df_task2['db_score'] >= t_val
    n_auto = m.sum()
    n_man  = (~m).sum()
    acc    = (df_task2.loc[m, 'db_pred'] == df_task2.loc[m, 'label']).mean() if n_auto > 0 else 0
    hrs    = n_auto * 5 / 60
    print(f"Threshold = {t_name}:")
    print(f"  Auto-tagged   : {n_auto}/{N2} ({n_auto/N2*100:.1f}%)")
    print(f"  Human review  : {n_man}/{N2}  ({n_man/N2*100:.1f}%)")
    print(f"  Auto accuracy : {acc*100:.1f}%")
    print(f"  Time saved    : ~{hrs:.1f} hours  (at 5 min/review)")
    print()


### Step 2.6 — Interactive Batch Analysis Widget

Use this widget to simulate ReviewIQ's pipeline on your own review text. Enter one review per line, adjust the confidence threshold, and observe which reviews would be auto-tagged and which would be sent to human review.

**Try entering the five sample reviews, then replace them with your own.**

**Reflection Question Q2** — answer in the Markdown cell below after experimenting.


In [ ]:
def run_batch_widget(text_block, threshold):
    lines = [l.strip() for l in text_block.strip().split('\n') if l.strip()]
    if not lines:
        print("Enter at least one review."); return

    results = sentiment_pipe(lines)

    fig, axes = plt.subplots(1, 2, figsize=(13, max(3.5, len(lines) * 0.65 + 2)))
    fig.suptitle(f"ReviewIQ Batch Analysis  (threshold = {threshold:.2f})",
                 fontsize=12, fontweight='bold')

    # Panel A: confidence bars
    ax1 = axes[0]
    scores = [r['score'] for r in results]
    preds  = [r['label'] for r in results]
    bar_colors = ['#70AD47' if p == 'POSITIVE' else '#FF4444' for p in preds]
    y_pos = np.arange(len(lines))
    ax1.barh(y_pos, scores, color=bar_colors, edgecolor='white', height=0.55)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels([f"Review {i+1}" for i in range(len(lines))], fontsize=9)
    ax1.set_xlim(0, 1.08)
    ax1.axvline(threshold, color='black', ls='--', alpha=0.7,
                label=f'Threshold ({threshold:.2f})')
    ax1.set_xlabel("Confidence Score")
    ax1.set_title("Sentiment Confidence")
    ax1.legend(fontsize=9)
    for i, (sc, pr) in enumerate(zip(scores, preds)):
        ax1.text(sc + 0.01, i, f"{pr} {sc:.0%}", va='center', fontsize=8)
    ax1.grid(axis='x', alpha=0.3)

    # Panel B: decision table
    ax2 = axes[1]
    ax2.axis('off')
    table_data = []
    for i, (line, res) in enumerate(zip(lines, results)):
        action = "AUTO ✓" if res['score'] >= threshold else "HUMAN ✎"
        table_data.append([f"#{i+1}", res['label'], f"{res['score']:.1%}", action])
    tbl = ax2.table(cellText=table_data,
                    colLabels=['#', 'Sentiment', 'Confidence', 'Action'],
                    loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.1, 1.8)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#404040'); cell.set_text_props(color='white')
        elif col == 3 and row > 0:
            val = table_data[row-1][3]
            cell.set_facecolor('#d4edda' if 'AUTO' in val else '#fff3cd')
    ax2.set_title("Auto-tagging Decisions", fontsize=10)

    plt.tight_layout()
    plt.show()

default_text = (
    "This product exceeded every expectation — absolutely love it!\n"
    "Arrived damaged and customer service was completely useless.\n"
    "It works fine, nothing special, would probably buy again.\n"
    "Five stars. Best purchase I have made all year.\n"
    "Stopped working after two weeks. Total waste of money."
)

text_area = widgets.Textarea(
    value=default_text,
    layout=widgets.Layout(width='65%', height='115px'),
)
thresh_slider = widgets.FloatSlider(
    value=0.90, min=0.50, max=0.99, step=0.01,
    description='Threshold:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='55%'),
)
run_btn2  = widgets.Button(description='Run Batch Analysis', button_style='primary',
                           layout=widgets.Layout(width='185px'))
out2 = widgets.Output()

def on_run2(b):
    out2.clear_output(wait=True)
    with out2:
        run_batch_widget(text_area.value, thresh_slider.value)

run_btn2.on_click(on_run2)
print("Step 2.6 — ReviewIQ Batch Analysis Widget")
print("Enter one review per line. Adjust threshold to control auto-tagging coverage.\n")
display(widgets.VBox([
    widgets.Label("Reviews (one per line):"),
    text_area,
    thresh_slider,
    run_btn2,
    out2,
]))


### Reflection Question Q2 — Answer Here

**Q2.** Using the batch widget in Step 2.6, run the five default reviews at threshold **0.50** and then at threshold **0.90**. Record your observations:

- At threshold 0.50: How many reviews are auto-tagged? What is the "Action" column for each?
- At threshold 0.90: How many reviews are auto-tagged? Which ones are routed to human review?
- In the Step 2.5 threshold chart, at threshold 0.90, what does the model's auto-tag accuracy become? What is the coverage trade-off (% of reviews still needing human review)?

*Answer: …*


---
## 3.10 — Conclusions

This notebook built a complete NLP sentiment analysis pipeline for the ReviewIQ / TechNest use case:

| | Task 1 — Comparison | Task 2 — Deployment |
|-|---------------------|---------------------|
| **Type** | Rule-based vs. Transformer | Full evaluation + threshold tuning |
| **Goal** | Understand tokenization and model evolution | Measure real accuracy; design production system |
| **Key concept** | WordPiece tokens, VADER lexicon, DistilBERT pipeline | Confusion matrix, calibration, confidence threshold |
| **Output** | Accuracy comparison + disagreement analysis | Auto-tagging system with tunable threshold |
| **Session** | 33 | 33 |

**Key takeaways:**
- Tokenization is the foundation of all transformer-based NLP — WordPiece handles unknown words gracefully by splitting them into subword units.
- Rule-based models (VADER) are fast and require no data, but fail on sarcasm, complex negation, and domain-specific language.
- Transformer models (DistilBERT) learn contextual representations that capture sentiment beyond individual word polarities — but they are not infallible.
- Confidence scores enable a **human-in-the-loop** deployment strategy: auto-tag high-confidence predictions and route uncertain ones to human review.
- Error analysis — not just accuracy — is the practitioner's tool for understanding model limitations and planning improvements.

---

## 3.11 — Reflection Questions

*Write your answers in the Markdown cells below each question.*

**Q3.** In Step 1.2, DistilBERT tokenizes text using **WordPiece**. The word *"electroencephalography"* is split into multiple subword tokens. Explain why subword tokenization is preferable to character-level or word-level tokenization for a model serving a global e-commerce platform that receives reviews in technical language, product model names, and informal abbreviations.

*Answer: …*

---

**Q4.** The Step 2.4 calibration chart shows that predictions in the **0.50–0.70 confidence bucket** have a different accuracy than those in the **0.90–1.00 bucket**. Look at your chart and answer:
- Are the low-confidence predictions more or less accurate than high-confidence ones?
- What does this tell you about whether DistilBERT is well-calibrated for this task?
- ReviewIQ is deciding whether to auto-tag reviews with confidence ≥ 0.70 or ≥ 0.90. Using specific numbers from your calibration chart, make a recommendation.

*Answer: …*

---

**Q5.** DistilBERT was fine-tuned on **SST-2** (movie reviews from Rotten Tomatoes), and we evaluated it on **IMDB** movie reviews. Suppose ReviewIQ now wants to deploy this pipeline on **TechNest's actual product reviews** (electronics, appliances, gadgets). Describe two reasons why the model's accuracy on TechNest reviews might differ from what we observed on IMDB, and explain what a data scientist would need to do to adapt the model to the new domain.

*Answer: …*
